# Notebook ETL (Extract, Transform, Load) - NCR Ride Bookings & Weather 2024

Notebook ini berisi alur kerja ETL yang sistematis untuk membersihkan, mentransformasikan, dan menggabungkan data perjalanan NCR (`ncr_ride_bookings.csv`) dengan data cuaca NCR tahun 2024 (`weather_ncr_2024_full.json`).

### Tahapan ETL:
1. **Extract**: Membaca dataset perjalanan (format CSV) dan data cuaca (format JSON).
2. **Clean**: Melakukan pembersihan tipe data, standarisasi teks, penanganan nilai kosong (missing values), dan ekstrak komponen waktu.
3. **Transform**: Memetakan lokasi pickup & drop ke stasiun kota cuaca utama, menggabungkan data perjalanan dan cuaca secara temporal & spasial, serta membuat kolom fitur kualitatif baru.
4. **Load**: Menyimpan semua dataset yang telah bersih dan gabungan ke dalam folder output `cleaned_data`.

In [12]:
import pandas as pd
import numpy as np
import json
import os
from pathlib import Path

# Pengaturan Pandas untuk visualisasi data yang rapi
pd.set_option('display.max_columns', None)
pd.set_option('display.float_format', '{:,.2f}'.format)

## 1. Extract & Clean - Data Pemesanan Perjalanan (Ride Bookings)

Pertama, kita muat dataset perjalanan dari file `ncr_ride_bookings.csv` dan memeriksa strukturnya.

In [13]:
ride_path = 'dataset/ncr_ride_bookings.csv'

print("Memuat data pemesanan...")
df_ride = pd.read_csv(ride_path)
print(f"Ukuran dataset pemesanan: {df_ride.shape}")
df_ride.head()

Memuat data pemesanan...
Ukuran dataset pemesanan: (150000, 21)


,Date,Time,Booking ID,Booking Status,Customer ID,Vehicle Type,Pickup Location,Drop Location,Avg VTAT,Avg CTAT,Cancelled Rides by Customer,Reason for cancelling by Customer,Cancelled Rides by Driver,Driver Cancellation Reason,Incomplete Rides,Incomplete Rides Reason,Booking Value,Ride Distance,Driver Ratings,Customer Rating,Payment Method
0,2024-03-23,12:29:38,"""CNR5884300""",No Driver Found,"""CID1982111""",eBike,Palam Vihar,Jhilmil,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
1,2024-11-29,18:01:39,"""CNR1326809""",Incomplete,"""CID4604802""",Go Sedan,Shastri Nagar,Gurgaon Sector 56,4.90,14.00,NaN,NaN,NaN,NaN,1.00,Vehicle Breakdown,237.00,5.73,NaN,NaN,UPI
2,2024-08-23,08:56:10,"""CNR8494506""",Completed,"""CID9202816""",Auto,Khandsa,Malviya Nagar,13.40,25.80,NaN,NaN,NaN,NaN,NaN,NaN,627.00,13.58,4.90,4.90,Debit Card
3,2024-10-21,17:17:25,"""CNR8906825""",Completed,"""CID2610914""",Premier Sedan,Central Secretariat,Inderlok,13.10,28.50,NaN,NaN,NaN,NaN,NaN,NaN,416.00,34.02,4.60,5.00,UPI
4,2024-09-16,22:08:00,"""CNR1950162""",Completed,"""CID9933542""",Bike,Ghitorni Village,Khan Market,5.30,19.60,NaN,NaN,NaN,NaN,NaN,NaN,737.00,48.21,4.10,4.30,UPI


### 1.1 Pembersihan Dasar Data Pemesanan
Langkah cleaning dasar meliputi:
- Menghapus spasi di awal/akhir teks untuk semua kolom objek.
- Menghapus tanda kutip ganda (`"`) pada kolom ID.
- Parsing kolom tanggal (`Date`) dan waktu (`Time`) menjadi objek datetime dan time.
- Membuat kolom gabungan `Booking Datetime`.
- Menstandarkan penamaan tipe kendaraan (mengubah `Uber XL` menjadi `UberXL`).
- Mengonversi kolom indikator pembatalan/ride tidak lengkap menjadi tipe `int64` dengan mengisi kosong sebagai `0`.
- Mengonversi kolom numerik lainnya ke tipe data numerik yang tepat.

In [14]:
df_ride_clean = df_ride.copy()

# 1. Bersihkan spasi berlebih pada kolom string
object_cols = df_ride_clean.select_dtypes(include='object').columns
for col in object_cols:
    df_ride_clean[col] = df_ride_clean[col].str.strip()

# 2. Bersihkan tanda kutip pada ID
df_ride_clean['Booking ID'] = df_ride_clean['Booking ID'].str.replace('"', '', regex=False)
df_ride_clean['Customer ID'] = df_ride_clean['Customer ID'].str.replace('"', '', regex=False)

# 3. Konversi format Tanggal dan Waktu
df_ride_clean['Date'] = pd.to_datetime(df_ride_clean['Date'], errors='coerce')
parsed_time = pd.to_datetime(df_ride_clean['Time'], format='%H:%M:%S', errors='coerce')
df_ride_clean['Time'] = parsed_time.dt.time
df_ride_clean['Booking Datetime'] = pd.to_datetime(
    df_ride_clean['Date'].dt.strftime('%Y-%m-%d') + ' ' + parsed_time.dt.strftime('%H:%M:%S'),
    errors='coerce'
)

# 4. Standarisasi nama kendaraan
df_ride_clean['Vehicle Type'] = df_ride_clean['Vehicle Type'].replace({'Uber XL': 'UberXL'})

# 5. Konversi kolom indikator biner ke integer
indicator_cols = ['Cancelled Rides by Customer', 'Cancelled Rides by Driver', 'Incomplete Rides']
for col in indicator_cols:
    df_ride_clean[col] = pd.to_numeric(df_ride_clean[col], errors='coerce').fillna(0).astype('int64')

# 6. Konversi kolom numerik
numeric_cols = ['Avg VTAT', 'Avg CTAT', 'Booking Value', 'Ride Distance', 'Driver Ratings', 'Customer Rating']
for col in numeric_cols:
    df_ride_clean[col] = pd.to_numeric(df_ride_clean[col], errors='coerce')

# 7. Ekstrak komponen waktu untuk agregasi analisis
df_ride_clean['Hour'] = df_ride_clean['Booking Datetime'].dt.hour
df_ride_clean['Month'] = df_ride_clean['Date'].dt.month
df_ride_clean['Month Name'] = df_ride_clean['Date'].dt.month_name()

print("Pembersihan dasar data ride bookings selesai.")
df_ride_clean.info()

Pembersihan dasar data ride bookings selesai.
<class 'pandas.core.frame.DataFrame'>
RangeIndex: 150000 entries, 0 to 149999
Data columns (total 25 columns):
 #   Column                             Non-Null Count   Dtype         
---  ------                             --------------   -----         
 0   Date                               150000 non-null  datetime64[ns]
 1   Time                               150000 non-null  object        
 2   Booking ID                         150000 non-null  object        
 3   Booking Status                     150000 non-null  object        
 4   Customer ID                        150000 non-null  object        
 5   Vehicle Type                       150000 non-null  object        
 6   Pickup Location                    150000 non-null  object        
 7   Drop Location                      150000 non-null  object        
 8   Avg VTAT                           139500 non-null  float64       
 9   Avg CTAT                           102000 non-

### 1.2 Analisis Nilai Kosong (Missing Values)
Mari kita cek seberapa banyak data kosong yang ada di dataset booking setelah pembersihan dasar.

In [15]:
missing_ride = df_ride_clean.isnull().sum()
print("Jumlah missing values per kolom:")
print(missing_ride[missing_ride > 0])

Jumlah missing values per kolom:
Avg VTAT                              10500
Avg CTAT                              48000
Reason for cancelling by Customer    139500
Driver Cancellation Reason           123000
Incomplete Rides Reason              141000
Booking Value                         48000
Ride Distance                         48000
Driver Ratings                        57000
Customer Rating                       57000
Payment Method                        48000
dtype: int64


**Catatan Bisnis & Pembersihan Lanjutan**:
- Nilai kosong pada `Booking Value` dan `Ride Distance` (48,000 baris) adalah pesanan yang **batal atau tidak terlaksana** (Booking Status != 'Completed'). Ini secara logika bisnis wajar karena tidak ada jarak dan nilai bayar untuk perjalanan yang tidak berjalan.
- Nilai kosong pada `Avg VTAT` dan `Avg CTAT` terjadi saat tidak ada driver yang merespon atau pesanan batal di awal.
- Nilai kosong pada `Driver Ratings` dan `Customer Rating` berarti customer/driver tidak mengisi rating setelah perjalanan.
Untuk menjaga akurasi analisis rating dan nilai bisnis, kita **membiarkan nilai kosong ini sebagai NaN** (tidak diimputasi) karena imputasi dummy (seperti mengisi rating dengan 0 atau rata-rata) akan mendistorsi statistik asli.

## 2. Extract & Clean - Data Cuaca (Weather Data)

Data cuaca NCR tahun 2024 disimpan dalam format JSON. Kita akan memuatnya, mengekstrak data dari array `data`, dan membersihkannya.

In [16]:
weather_path = 'dataset/weather_ncr_2024_full.json'

print("Memuat data cuaca...")
with open(weather_path, 'r', encoding='utf-8') as f:
    weather_json = json.load(f)

# Load list dari data
df_weather = pd.DataFrame(weather_json['data'])
print(f"Ukuran data cuaca mentah: {df_weather.shape}")
df_weather.head()
df_weather.info()

Memuat data cuaca...
Ukuran data cuaca mentah: (61488, 22)
<class 'pandas.core.frame.DataFrame'>
RangeIndex: 61488 entries, 0 to 61487
Data columns (total 22 columns):
 #   Column           Non-Null Count  Dtype  
---  ------           --------------  -----  
 0   date             61488 non-null  object 
 1   time             61488 non-null  object 
 2   hour             61488 non-null  int64  
 3   location         61488 non-null  object 
 4   temperature_c    61488 non-null  float64
 5   feelslike_c      61488 non-null  float64
 6   dew_point_c      61488 non-null  float64
 7   humidity_pct     61488 non-null  float64
 8   pressure_hpa     61488 non-null  float64
 9   visibility_km    61432 non-null  float64
 10  cloudcover_pct   61488 non-null  float64
 11  uvindex          61488 non-null  float64
 12  rainfall_mm      61488 non-null  float64
 13  precip_prob_pct  61488 non-null  float64
 14  precip_type      1241 non-null   object 
 15  windspeed_kmh    61488 non-null  float64
 16 

### 2.1 Pembersihan Data Cuaca
- Konversi kolom `date` dan `time` ke tipe data waktu yang sesuai.
- Ubah kolom `hour` menjadi integer.
- Konversi semua kolom parameter cuaca ke float.
- Imputasi nilai kosong jika ada, misalnya pada kolom `visibility_km` menggunakan nilai median grup lokasi dan jam.

In [17]:
df_weather_clean = df_weather.copy()

# 1. Konversi Tanggal dan Waktu
df_weather_clean['date'] = pd.to_datetime(df_weather_clean['date'], errors='coerce')
parsed_w_time = pd.to_datetime(df_weather_clean['time'], format='%H:%M:%S', errors='coerce')
df_weather_clean['time'] = parsed_w_time.dt.time
df_weather_clean['hour'] = pd.to_numeric(df_weather_clean['hour'], errors='coerce').astype('int64')

# 2. Konversi kolom cuaca numerik ke float
weather_num_cols = [
    'temperature_c', 'feelslike_c', 'dew_point_c', 'humidity_pct',
    'pressure_hpa', 'visibility_km', 'cloudcover_pct', 'uvindex',
    'rainfall_mm', 'precip_prob_pct', 'windspeed_kmh', 'winddir_deg', 'windgust_kmh'
]
for col in weather_num_cols:
    df_weather_clean[col] = pd.to_numeric(df_weather_clean[col], errors='coerce')

# 3. Cek missing values
print("Missing values sebelum imputasi:")
print(df_weather_clean[weather_num_cols].isnull().sum())

# 4. Imputasi visibility_km yang kosong menggunakan median berdasarkan lokasi dan jam
df_weather_clean['visibility_km'] = df_weather_clean.groupby(['location', 'hour'])['visibility_km'].transform(
    lambda x: x.fillna(x.median())
)
# Jika masih ada sisa kosong, isi dengan median keseluruhan
df_weather_clean['visibility_km'] = df_weather_clean['visibility_km'].fillna(df_weather_clean['visibility_km'].median())

print("\nMissing values setelah imputasi:")
print(df_weather_clean[weather_num_cols].isnull().sum())

Missing values sebelum imputasi:
temperature_c       0
feelslike_c         0
dew_point_c         0
humidity_pct        0
pressure_hpa        0
visibility_km      56
cloudcover_pct      0
uvindex             0
rainfall_mm         0
precip_prob_pct     0
windspeed_kmh       0
winddir_deg         0
windgust_kmh        0
dtype: int64

Missing values setelah imputasi:
temperature_c      0
feelslike_c        0
dew_point_c        0
humidity_pct       0
pressure_hpa       0
visibility_km      0
cloudcover_pct     0
uvindex            0
rainfall_mm        0
precip_prob_pct    0
windspeed_kmh      0
winddir_deg        0
windgust_kmh       0
dtype: int64


## 3. Transformasi Data (Transformation)

### 3.1 Pemetaan Geografis: Menghubungkan Lokasi Booking dengan Stasiun Cuaca Kota
Pemesanan memiliki 176 lokasi penjemputan/penurunan yang spesifik (seperti "Cyber Hub", "Noida Sector 62"). Di sisi lain, data cuaca tercatat di 7 stasiun kota utama di wilayah NCR: Delhi, Gurgaon, Noida, Faridabad, Ghaziabad, Sonipat, dan Meerut.

Kita akan memetakan setiap lokasi ke salah satu dari stasiun cuaca kota utama tersebut berdasarkan kata kunci geografis agar datanya dapat digabungkan secara spasial.

In [18]:
# Tentukan kata kunci untuk mencocokkan nama kota
gurgaon_keywords = [
    'gurgaon', 'manesar', 'udyog vihar', 'dlf', 'cyber hub', 'sikanderpur',
    'huda city centre', 'iffco chowk', 'sohna road', 'badshahpur', 'palam vihar',
    'ardee city', 'sushant lok', 'ambience mall', 'hero honda chowk', 'vatika chowk',
    'subhash chowk', 'khandsa', 'narsinghpur', 'gwal pahari', 'kadarpur', 'basai dhankot',
    'mg road', 'pataudi chowk', 'rajiv nagar', 'new colony', 'sadar bazar gurgaon',
    'golf course road', 'kherki daula toll', 'bhiwadi'
]
noida_keywords = ['noida', 'botanical garden']
faridabad_keywords = ['faridabad']
ghaziabad_keywords = ['ghaziabad', 'indirapuram', 'kaushambi', 'vaishali', 'raj nagar extension']
sonipat_keywords = ['sonipat', 'panipat']
meerut_keywords = ['meerut']

def map_location_to_city(loc):
    if not isinstance(loc, str):
        return 'Delhi'
    loc_lower = loc.lower()
    for kw in gurgaon_keywords:
        if kw in loc_lower:
            return 'Gurgaon'
    for kw in noida_keywords:
        if kw in loc_lower:
            return 'Noida'
    for kw in faridabad_keywords:
        if kw in loc_lower:
            return 'Faridabad'
    for kw in ghaziabad_keywords:
        if kw in loc_lower:
            return 'Ghaziabad'
    for kw in sonipat_keywords:
        if kw in loc_lower:
            return 'Sonipat'
    for kw in meerut_keywords:
        if kw in loc_lower:
            return 'Meerut'
    return 'Delhi'

df_ride_clean['Pickup City'] = df_ride_clean['Pickup Location'].apply(map_location_to_city)
df_ride_clean['Drop City'] = df_ride_clean['Drop Location'].apply(map_location_to_city)

print("Hasil Pemetaan Kota (Pickup City):")
print(df_ride_clean['Pickup City'].value_counts())

Hasil Pemetaan Kota (Pickup City):
Pickup City
Delhi        105787
Gurgaon       31502
Noida          5072
Ghaziabad      4274
Sonipat        1700
Meerut          834
Faridabad       831
Name: count, dtype: int64


### 3.2 Penggabungan Dataset secara Spatio-Temporal
Gabungkan data perjalanan dengan data cuaca dengan mencocokkan:
- Tanggal perjalanan (`Date` == `date`)
- Jam perjalanan (`Hour` == `hour`)
- Lokasi awal perjalanan (`Pickup City` == `location`)

Kami menggunakan Left Join agar semua transaksi booking perjalanan tetap utuh.

In [19]:
# Siapkan dataframe cuaca dengan nama kolom yang cocok untuk merging
df_weather_merge = df_weather_clean.rename(columns={
    'date': 'Date',
    'hour': 'Hour',
    'location': 'Pickup City'
})

# Tentukan kolom cuaca penting untuk analisis
weather_cols = [
    'Date', 'Hour', 'Pickup City', 'temperature_c', 'feelslike_c', 
    'humidity_pct', 'visibility_km', 'rainfall_mm', 'windspeed_kmh', 
    'conditions', 'description', 'icon'
]

df_weather_subset = df_weather_merge[weather_cols]

# Lakukan merge
df_merged = pd.merge(
    df_ride_clean,
    df_weather_subset,
    on=['Date', 'Hour', 'Pickup City'],
    how='left'
)

print(f"Ukuran dataset tergabung: {df_merged.shape}")
print("Jumlah nilai null pada kolom cuaca setelah digabungkan:")
print(df_merged[weather_cols[3:]].isnull().sum())

Ukuran dataset tergabung: (150000, 36)
Jumlah nilai null pada kolom cuaca setelah digabungkan:
temperature_c    0
feelslike_c      0
humidity_pct     0
visibility_km    0
rainfall_mm      0
windspeed_kmh    0
conditions       0
description      0
icon             0
dtype: int64


### 3.3 Pembuatan Fitur Baru (Feature Engineering)
Untuk mempermudah visualisasi dan interpretasi data selama analisis, kita membuat 3 kolom kategori kualitatif baru berbasis nilai numerik cuaca:
1. `Temperature Category`: Mengelompokkan suhu menjadi Dingin, Sedang, atau Panas.
2. `Rain Category`: Mengelompokkan curah hujan menjadi Tidak Hujan, Hujan Ringan, atau Hujan Lebat.
3. `Visibility Category`: Mengelompokkan visibilitas menjadi Buruk/Kabut, Sedang, atau Baik.

In [20]:
# 1. Kategori Suhu
def get_temp_category(t):
    if t < 15:
        return 'Dingin (<15°C)'
    elif t <= 30:
        return 'Sedang (15°C - 30°C)'
    else:
        return 'Panas (>30°C)'

df_merged['Temperature Category'] = df_merged['temperature_c'].apply(get_temp_category)

# 2. Kategori Hujan
def get_rain_category(r):
    if r == 0:
        return 'Tidak Hujan'
    elif r <= 2.5:
        return 'Hujan Ringan (<=2.5mm/jam)'
    else:
        return 'Hujan Lebat (>2.5mm/jam)'

df_merged['Rain Category'] = df_merged['rainfall_mm'].apply(get_rain_category)

# 3. Kategori Jarak Pandang
def get_visibility_category(v):
    if v < 2:
        return 'Buruk/Kabut (<2km)'
    elif v <= 5:
        return 'Sedang (2km - 5km)'
    else:
        return 'Baik (>5km)'

df_merged['Visibility Category'] = df_merged['visibility_km'].apply(get_visibility_category)

print("Feature engineering selesai. Sampel data baru:")
df_merged[[
    'Booking ID', 'Pickup Location', 'Pickup City', 'temperature_c', 
    'Temperature Category', 'rainfall_mm', 'Rain Category', 'visibility_km', 'Visibility Category'
]].head()

Feature engineering selesai. Sampel data baru:


,Booking ID,Pickup Location,Pickup City,temperature_c,Temperature Category,rainfall_mm,Rain Category,visibility_km,Visibility Category
0,CNR5884300,Palam Vihar,Gurgaon,32.00,Panas (>30°C),0.00,Tidak Hujan,4.00,Sedang (2km - 5km)
1,CNR1326809,Shastri Nagar,Delhi,20.00,Sedang (15°C - 30°C),0.00,Tidak Hujan,3.00,Sedang (2km - 5km)
2,CNR8494506,Khandsa,Gurgaon,28.90,Sedang (15°C - 30°C),0.00,Tidak Hujan,3.70,Sedang (2km - 5km)
3,CNR8906825,Central Secretariat,Delhi,31.10,Panas (>30°C),0.00,Tidak Hujan,2.20,Sedang (2km - 5km)
4,CNR1950162,Ghitorni Village,Delhi,28.00,Sedang (15°C - 30°C),0.00,Tidak Hujan,4.00,Sedang (2km - 5km)


## 4. Load (Penyimpanan Data Bersih)

Langkah terakhir adalah menyimpan file yang telah dibersihkan dan ditransformasikan ke dalam folder output `cleaned_data`.

In [21]:
output_dir = 'cleaned_data'
os.makedirs(output_dir, exist_ok=True)

print("Menyimpan file-file bersih...")
df_ride_clean.to_csv(os.path.join(output_dir, 'ride_bookings_cleaned.csv'), index=False)
df_weather_clean.to_csv(os.path.join(output_dir, 'weather_cleaned.csv'), index=False)
df_merged.to_csv(os.path.join(output_dir, 'ride_bookings_weather_merged.csv'), index=False)

print("\n=== PROSES ETL SELESAI ===")
print(f"Data bersih tersimpan di folder '{output_dir}':")
print(f"- Booking Perjalanan Bersih: {os.path.join(output_dir, 'ride_bookings_cleaned.csv')}")
print(f"- Data Cuaca Bersih: {os.path.join(output_dir, 'weather_cleaned.csv')}")
print(f"- Data Merged (Gabungan): {os.path.join(output_dir, 'ride_bookings_weather_merged.csv')}")

Menyimpan file-file bersih...



=== PROSES ETL SELESAI ===
Data bersih tersimpan di folder 'cleaned_data':
- Booking Perjalanan Bersih: cleaned_data\ride_bookings_cleaned.csv
- Data Cuaca Bersih: cleaned_data\weather_cleaned.csv
- Data Merged (Gabungan): cleaned_data\ride_bookings_weather_merged.csv


## Langkah 5: Load ke PostgreSQL

Bagian ini memuat data yang telah dibersihkan ke dalam database PostgreSQL baru bernama `uas_bigdata` menggunakan kredensial koneksi Anda.

In [27]:
import psycopg2
from psycopg2.extensions import ISOLATION_LEVEL_AUTOCOMMIT
from sqlalchemy import create_engine
import pandas as pd
import os

# Konfigurasi Koneksi PostgreSQL
DB_USER = 'postgres'
DB_PASSWORD = 'uas_bigdata2026'
DB_HOST = 'db.okculcsrmwiactymkqzq.supabase.co'
DB_PORT = 5432
DB_NAME = 'postgres'

print("Menghubungkan ke PostgreSQL untuk membuat database baru...")
# Koneksi awal ke database default 'postgres'
conn = psycopg2.connect(
    dbname='postgres',
    user=DB_USER,
    password=DB_PASSWORD,
    host=DB_HOST,
    port=DB_PORT,
    sslmode='require'
)
conn.set_isolation_level(ISOLATION_LEVEL_AUTOCOMMIT)
cursor = conn.cursor()

# Cek apakah database uas_bigdata sudah ada
cursor.execute(f"SELECT 1 FROM pg_catalog.pg_database WHERE datname = '{DB_NAME}'")
exists = cursor.fetchone()
if not exists:
    cursor.execute(f"CREATE DATABASE {DB_NAME}")
    print(f"Database '{DB_NAME}' berhasil dibuat.")
else:
    print(f"Database '{DB_NAME}' sudah ada.")

cursor.close()
conn.close()

# Koneksi ke database baru menggunakan SQLAlchemy
print(f"Mengunggah data ke database '{DB_NAME}'...")
engine = create_engine(f"postgresql+psycopg2://{DB_USER}:{DB_PASSWORD}@{DB_HOST}:{DB_PORT}/{DB_NAME}?sslmode=require")

output_dir = 'cleaned_data'
files_to_load = {
    'ride_bookings_weather_merged': 'ride_bookings_weather_merged.csv'
}

for table_name, file_name in files_to_load.items():
    file_path = os.path.join(output_dir, file_name)
    if os.path.exists(file_path):
        print(f"Memuat {file_name} ke tabel '{table_name}'...")
        df = pd.read_csv(file_path)
        df.to_sql(name=table_name, con=engine, if_exists='replace', index=False)
        print(f"Selesai! {len(df)} baris berhasil dimuat ke '{table_name}'.")
    else:
        print(f"File tidak ditemukan: {file_path}")

print("\n=== SEMUA DATA BERHASIL DIMUAT KE POSTGRESQL ===")

Menghubungkan ke PostgreSQL untuk membuat database baru...
Database 'postgres' sudah ada.
Mengunggah data ke database 'postgres'...
Memuat ride_bookings_weather_merged.csv ke tabel 'ride_bookings_weather_merged'...
Selesai! 150000 baris berhasil dimuat ke 'ride_bookings_weather_merged'.

=== SEMUA DATA BERHASIL DIMUAT KE POSTGRESQL ===
